In [ ]:
# ------------------------------------
# OLD CODE
# ------------------------------------
# import glob
# import numpy as np
# import librosa

# segment_dur = 2
# directories = ['guitar', 'non_guitar']
# def load_segments(directory):
#     segments = []
#     for path in glob.glob(f'{directory}/*.wav'):
#         signal, sr = librosa.load(path, sr=None, mono=True)
#         seg_len = sr * segment_dur
#         audio_segments = librosa.util.frame(signal, frame_length=seg_len, hop_length=seg_len, axis=0)
#         segments.append(audio_segments)
#     return np.concatenate(segments)

# guitar_segments = load_segments('guitar')
# non_guitar_segments = load_segments('non_guitar')
# def get_chords(path):

#     chords = []
#     with open(path) as f:
#         for line in f:      
#             start_s, end_s, chord = line.strip().split('\t')
#             chords.append(chord)
#     return chords


# guitar_chords = []
# for i in range(6):
#     guitar_chords += get_chords(f'guitar/guitar_annotation.lab')

# non_guitar_chords = []
# for i in range(10):
#     non_guitar_chords += get_chords(f'non_guitar/non_guitar_annotation.lab')

/Applications/anaconda3/envs/ai-py311/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
#FEATURES OPTION 1: chroma (works well when the inversions are off)
#only need to run one of the 2 features cells
import numpy as np, librosa

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32) #loads the audio clips and converts them to float32 for librosa

picture = librosa.feature.chroma_stft(y=X_audio, sr=sr, n_fft=8192, hop_length=4096) #the 12 chroma bins across 8 time frames for each clip (the CNNs use this one)
X = np.concatenate([picture.mean(axis=-1), picture.std(axis=-1)], axis=1) #the mean and std of the 12 chroma bins, so 24 features (everything else uses this one)

features_used = "chroma" 
print(features_used, X.shape, picture.shape)

chroma (7398, 24) (7398, 12, 8)


In [ ]:
#FEATURES OPTION 2: wide spectrum (much better for the MLP, CNN and logistic regression when the inversions are on, but bad for k-NN)
#chroma squashes all the octaves together so it cant tell which note is the lowest one, and thats the whole point of an inversion
#the CQT keeps every note seperate, so its 84 notes (C1 up to B7) instead of just 12       
import numpy as np, librosa

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32) #loads the audio clips and converts them to float32 for librosa

cqt = np.abs(librosa.cqt(y=X_audio, sr=sr, hop_length=4096, fmin=librosa.note_to_hz("C1"), n_bins=84))
cqt = cqt / cqt.max(axis=(1, 2), keepdims=True) #the max of each clip separately
picture = np.log1p(100 * cqt)
X = picture.mean(axis=-1) 

features_used = "wide spectrum" 
print(features_used, X.shape, picture.shape)

wide spectrum (7398, 84) (7398, 84, 8)


In [9]:
#makes the train, validation and test sets out of whichever features i ran last
#(if i change the features or the inversions i have to run this again, and then run the models again)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

INCLUDE_INVERSIONS = False #set this to True to keep the inversions in the labels, or False to drop them

labels = pd.read_csv("data/processed/labels.csv")

if INCLUDE_INVERSIONS == True:
    y = labels["original_label"].to_numpy() #keeps the label how it is (e.g. 'C:maj/3') -> 288 classes
else:
    y = labels["original_label"].str.split("/").str[0].to_numpy()
    # Labels: drop the inversion (e.g. 'C:maj/3' -> 'C:maj') -> 84 classes (12 roots x 7 qualities)

print('using', features_used, 'features')


#Split the data into training, validation and testing sets (70% train / 15% val / 15% test), same as stage 4
#picture goes in here as well so that its rows still line up with X and y after the split
X_train, X_rem, picture_train, picture_rem, y_train, y_rem = train_test_split(X, picture, y, test_size=0.3, stratify=y, random_state=42) #holds 30% aside for val/test
X_val, X_test, picture_val, picture_test, y_val, y_test = train_test_split(X_rem, picture_rem, y_rem, test_size=0.5, stratify=y_rem, random_state=42) #splits that remaining 30% evenly into 15% validation and 15% test


scaler = StandardScaler() #i will use this to scale the features before feeding them into the models, this is important because some models are sensitive to the scale of the features
scaler.fit(X_train) #learns the average and standard deviation from the training data only
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val) #scales the validation set using the training mean and std
X_test = scaler.transform(X_test)


#the current format of the labels is "C:maj" (or "C:maj/3" with inversions), i need to convert them to 2 seperate plain lists
#this means i can train the models on the root and quality seperately
#if the inversions are included then they just stay as part of the quality (e.g. 'maj/3'), so thats 24 qualities instead of 7

y_train_root = [label.split(":")[0] for label in y_train] #the root is the bit before the :
y_train_quality = [label.split(":")[1] for label in y_train] #the quality is the bit after the :

y_val_root = [label.split(":")[0] for label in y_val] #same for the validation set
y_val_quality = [label.split(":")[1] for label in y_val]

y_test_root = [label.split(":")[0] for label in y_test] #same for the test set
y_test_quality = [label.split(":")[1] for label in y_test]

print(len(set(y_train_root)), 'roots,', len(set(y_train_quality)), 'qualities')

using wide spectrum features
12 roots, 7 qualities


In [13]:
#Tune k-NN on the validation set before testing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

best_acc = 0
best_k = 0
for k in range(1,101):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train) #fit k-NN on the scaled training data
    val_pred = knn.predict(X_val) #test this k on the validation set
    acc = accuracy_score(y_val, val_pred)
    if acc > best_acc:
        best_acc = acc
        best_k = k

print(f"Best k for k-NN: {best_k} (val accuracy: {best_acc:.3f})\n")

Best k for k-NN: 79 (val accuracy: 0.175)



In [ ]:
#the list of models, each one predicts the whole chord in one go
#(run the k-NN cell first because this one needs best_k)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "k-NN": KNeighborsClassifier(n_neighbors=best_k),
    "SVM": CalibratedClassifierCV(SVC(C=10)),#C=10 improves accuracy and probability=True will let the demo show confidence bars later [note that i had to use CalibratedClassifierCV because of some update thingy
    "Random Forest": RandomForestClassifier(random_state=42)
}   #here ive made a dictionary to hold the four models, with their names as keys and the models as values, this will make it easier to loop through them for evaluation


for name, model in models.items():
    model.fit(X_train, y_train) #fit the model to the training data
    y_pred = model.predict(X_test) #predict the labels for the test data
    accuracy = accuracy_score(y_test, y_pred) #calculate the accuracy of the model, this is done by comparing the predicted labels with the true labels for the test data
    print(f"{name}: {accuracy:.3f}")

Logistic Regression: 0.771
k-NN: 0.160
SVM: 0.602
Random Forest: 0.523


In [22]:
#same list of models as before, but now each one is split into 2 models, one for the root and one for the quality
#(run the k-NN cell first because this one needs best_k)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

#these ones only predict the root
root_models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "k-NN": KNeighborsClassifier(n_neighbors=best_k),
    "SVM": CalibratedClassifierCV(SVC(C=10), ensemble=False),
    "Random Forest": RandomForestClassifier(random_state=42)
}

#these ones only predict the quality
quality_models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "k-NN": KNeighborsClassifier(n_neighbors=best_k),
    "SVM": CalibratedClassifierCV(SVC(C=10), ensemble=False),
    "Random Forest": RandomForestClassifier(random_state=42)
}


for name in root_models:
    root_model = root_models[name]
    quality_model = quality_models[name]

    root_model.fit(X_train, y_train_root) #trains the root model
    quality_model.fit(X_train, y_train_quality) #trains the quality model

    root_pred = root_model.predict(X_test) #predict the root labels for the test data
    quality_pred = quality_model.predict(X_test) #predict the quality labels for the test data
    combined_pred = [r + ":" + q for r, q in zip(root_pred, quality_pred)] #sticks the root and quality back together into a full chord

    print(name)
    print(f"root accuracy: {accuracy_score(y_test_root, root_pred):.3f}")
    print(f"quality accuracy: {accuracy_score(y_test_quality, quality_pred):.3f}")
    print(f"combined: {accuracy_score(y_test, combined_pred):.3f}\n") #only counts as correct if both the root and the quality are right

Logistic Regression
root accuracy: 0.784
quality accuracy: 0.197
combined: 0.142

k-NN
root accuracy: 0.508
quality accuracy: 0.181
combined: 0.083

SVM
root accuracy: 0.908
quality accuracy: 0.434
combined: 0.396

Random Forest
root accuracy: 0.804
quality accuracy: 0.376
combined: 0.313



In [ ]:
#basic pytorch MLP, it predicts the whole chord in one go using the same features as the other models
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42) #so that i get the same result every time i run it


#pytorch doesnt understand labels like 'C:maj', it needs numbers, so each chord gets a number (its position in this list)
chord_names = sorted(set(y_train))

y_train_numbers = []
for label in y_train:
    y_train_numbers.append(chord_names.index(label))

y_val_numbers = []
for label in y_val:
    y_val_numbers.append(chord_names.index(label))

y_test_numbers = []
for label in y_test:
    y_test_numbers.append(chord_names.index(label))


#turns everything into tensors, which are just pytorch's version of arrays
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_numbers)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) #gives me the training data in shuffled batches of 64


#the actual model, the features go in and 1 score per chord comes out
mlp = nn.Sequential(
    nn.Linear(X_train.shape[1], 128), #X_train.shape[1] is the number of features (24 for chroma, 84 for wide spectrum)
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, len(chord_names))
)

loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001) #this is what updates the weights
epochs = 60 #how many times it goes through the whole training set


for epoch in range(epochs):
    mlp.train() #puts the model in training mode
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad() #clears the gradients from the last batch
        output = mlp(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 10 == 0:
        mlp.eval() #puts the model in testing mode
        with torch.no_grad(): #dont need gradients when im only predicting
            val_output = mlp(X_val_tensor)
        val_pred = torch.argmax(val_output, dim=1) #picks the chord with the highest score
        val_accuracy = accuracy_score(y_val_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#now test it on the test set
mlp.eval()
with torch.no_grad():
    test_output = mlp(X_test_tensor)
test_pred = torch.argmax(test_output, dim=1)

print(f"\nMLP: {accuracy_score(y_test_numbers, test_pred.numpy()):.3f}")

In [8]:
#basic pytorch CNN, it also predicts the whole chord in one go
#instead of the averaged features it looks at the whole picture (12 notes tall for chroma or 84 notes tall for wide spectrum, and 8 time frames wide)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42) #so that i get the same result every time i run it


#same as the MLP, each chord gets a number (its position in this list)
chord_names = sorted(set(y_train))

y_train_numbers = []
for label in y_train:
    y_train_numbers.append(chord_names.index(label))

y_val_numbers = []
for label in y_val:
    y_val_numbers.append(chord_names.index(label))

y_test_numbers = []
for label in y_test:
    y_test_numbers.append(chord_names.index(label))


#the pictures are already between 0 and 1 so i dont need to scale them
#unsqueeze(1) adds the channel dimension, because the CNN wants (clips, channels, height, width) so it goes from (clips, notes, 8) to (clips, 1, notes, 8)
picture_train_tensor = torch.tensor(picture_train, dtype=torch.float32).unsqueeze(1)
picture_val_tensor = torch.tensor(picture_val, dtype=torch.float32).unsqueeze(1)
picture_test_tensor = torch.tensor(picture_test, dtype=torch.float32).unsqueeze(1)
y_train_tensor = torch.tensor(y_train_numbers)

train_data = TensorDataset(picture_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True) #gives me the training data in shuffled batches of 64


#the actual model
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), #1 channel in, 16 out, padding=1 keeps it the same size
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #only shrinks the time (8 -> 4), i dont want to shrink the notes because then it cant tell the roots apart
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #shrinks the time again (4 -> 2)
    nn.Flatten(), #turns it into one long list of 32 channels x notes x 2 time frames
    nn.Linear(32 * picture_train.shape[1] * 2, 128), #picture_train.shape[1] is the number of notes (12 or 84)
    nn.ReLU(),
    nn.Linear(128, len(chord_names))
)

loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001) #this is what updates the weights
epochs = 60 #how many times it goes through the whole training set


for epoch in range(epochs):
    cnn.train() #puts the model in training mode
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad() #clears the gradients from the last batch
        output = cnn(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 10 == 0:
        cnn.eval() #puts the model in testing mode
        with torch.no_grad(): #dont need gradients when im only predicting
            val_output = cnn(picture_val_tensor)
        val_pred = torch.argmax(val_output, dim=1) #picks the chord with the highest score
        val_accuracy = accuracy_score(y_val_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#now test it on the test set
cnn.eval()
with torch.no_grad():
    test_output = cnn(picture_test_tensor)
test_pred = torch.argmax(test_output, dim=1)

print(f"\nCNN: {accuracy_score(y_test_numbers, test_pred.numpy()):.3f}")

epoch 10: loss 0.053, val accuracy 0.868
epoch 20: loss 0.016, val accuracy 0.886
epoch 30: loss 0.008, val accuracy 0.888
epoch 40: loss 0.048, val accuracy 0.886
epoch 50: loss 0.019, val accuracy 0.897
epoch 60: loss 0.015, val accuracy 0.895

CNN: 0.893


In [9]:
#same MLP as before, but split into 2 models, one for the root and one for the quality
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42) #so that i get the same result every time i run it


#each root gets a number and each quality gets a number (its position in these lists)
root_names = sorted(set(y_train_root))
quality_names = sorted(set(y_train_quality))

y_train_root_numbers = []
for label in y_train_root:
    y_train_root_numbers.append(root_names.index(label))

y_train_quality_numbers = []
for label in y_train_quality:
    y_train_quality_numbers.append(quality_names.index(label))

y_val_root_numbers = []
for label in y_val_root:
    y_val_root_numbers.append(root_names.index(label))

y_val_quality_numbers = []
for label in y_val_quality:
    y_val_quality_numbers.append(quality_names.index(label))


#turns everything into tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_root_tensor = torch.tensor(y_train_root_numbers)
y_train_quality_tensor = torch.tensor(y_train_quality_numbers)

loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are
epochs = 60 #how many times it goes through the whole training set


#-------the root model-------
root_data = TensorDataset(X_train_tensor, y_train_root_tensor)
root_loader = DataLoader(root_data, batch_size=64, shuffle=True)

mlp_root = nn.Sequential(
    nn.Linear(X_train.shape[1], 128), #the number of features (24 for chroma, 84 for wide spectrum)
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, len(root_names)) #1 score per root
)

root_optimizer = torch.optim.Adam(mlp_root.parameters(), lr=0.001)

print("training the root model")
for epoch in range(epochs):
    mlp_root.train()
    for batch_X, batch_y in root_loader:
        root_optimizer.zero_grad() #clears the gradients from the last batch
        output = mlp_root(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        root_optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 10 == 0:
        mlp_root.eval()
        with torch.no_grad():
            val_output = mlp_root(X_val_tensor)
        val_pred = torch.argmax(val_output, dim=1)
        val_accuracy = accuracy_score(y_val_root_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#-------the quality model-------
quality_data = TensorDataset(X_train_tensor, y_train_quality_tensor)
quality_loader = DataLoader(quality_data, batch_size=64, shuffle=True)

mlp_quality = nn.Sequential(
    nn.Linear(X_train.shape[1], 128), #the number of features (24 for chroma, 84 for wide spectrum)
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, len(quality_names)) #1 score per quality
)

quality_optimizer = torch.optim.Adam(mlp_quality.parameters(), lr=0.001)

print("\ntraining the quality model")
for epoch in range(epochs):
    mlp_quality.train()
    for batch_X, batch_y in quality_loader:
        quality_optimizer.zero_grad() #clears the gradients from the last batch
        output = mlp_quality(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        quality_optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 10 == 0:
        mlp_quality.eval()
        with torch.no_grad():
            val_output = mlp_quality(X_val_tensor)
        val_pred = torch.argmax(val_output, dim=1)
        val_accuracy = accuracy_score(y_val_quality_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#-------now test both of them on the test set-------
mlp_root.eval()
mlp_quality.eval()
with torch.no_grad():
    root_output = mlp_root(X_test_tensor)
    quality_output = mlp_quality(X_test_tensor)
root_pred_numbers = torch.argmax(root_output, dim=1)
quality_pred_numbers = torch.argmax(quality_output, dim=1)

#turn the numbers back into the names, and stick them back together into full chords
root_pred = []
quality_pred = []
combined_pred = []
for i in range(len(y_test)):
    root = root_names[root_pred_numbers[i]]
    quality = quality_names[quality_pred_numbers[i]]
    root_pred.append(root)
    quality_pred.append(quality)
    combined_pred.append(root + ":" + quality)

print("\nMLP (2 models)")
print(f"root accuracy: {accuracy_score(y_test_root, root_pred):.3f}")
print(f"quality accuracy: {accuracy_score(y_test_quality, quality_pred):.3f}")
print(f"combined: {accuracy_score(y_test, combined_pred):.3f}") #only counts as correct if both the root and the quality are right

training the root model
epoch 10: loss 0.261, val accuracy 0.821
epoch 20: loss 0.103, val accuracy 0.862
epoch 30: loss 0.034, val accuracy 0.873
epoch 40: loss 0.125, val accuracy 0.886
epoch 50: loss 0.024, val accuracy 0.881
epoch 60: loss 0.141, val accuracy 0.886

training the quality model
epoch 10: loss 0.854, val accuracy 0.519
epoch 20: loss 0.518, val accuracy 0.644
epoch 30: loss 0.372, val accuracy 0.681
epoch 40: loss 0.223, val accuracy 0.729
epoch 50: loss 0.080, val accuracy 0.728
epoch 60: loss 0.118, val accuracy 0.733

MLP (2 models)
root accuracy: 0.885
quality accuracy: 0.743
combined: 0.667


In [7]:
#same CNN as before, but split into 2 models, one for the root and one for the quality
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42) #so that i get the same result every time i run it


#each root gets a number and each quality gets a number (its position in these lists)
root_names = sorted(set(y_train_root))
quality_names = sorted(set(y_train_quality))

y_train_root_numbers = []
for label in y_train_root:
    y_train_root_numbers.append(root_names.index(label))

y_train_quality_numbers = []
for label in y_train_quality:
    y_train_quality_numbers.append(quality_names.index(label))

y_val_root_numbers = []
for label in y_val_root:
    y_val_root_numbers.append(root_names.index(label))

y_val_quality_numbers = []
for label in y_val_quality:
    y_val_quality_numbers.append(quality_names.index(label))


#turns everything into tensors, unsqueeze(1) adds the channel dimension so it goes from (clips, notes, 8) to (clips, 1, notes, 8)
picture_train_tensor = torch.tensor(picture_train, dtype=torch.float32).unsqueeze(1)
picture_val_tensor = torch.tensor(picture_val, dtype=torch.float32).unsqueeze(1)
picture_test_tensor = torch.tensor(picture_test, dtype=torch.float32).unsqueeze(1)
y_train_root_tensor = torch.tensor(y_train_root_numbers)
y_train_quality_tensor = torch.tensor(y_train_quality_numbers)

loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are
epochs = 60 #how many times it goes through the whole training set


#-------the root model-------
root_data = TensorDataset(picture_train_tensor, y_train_root_tensor)
root_loader = DataLoader(root_data, batch_size=64, shuffle=True)

cnn_root = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #only shrinks the time (8 -> 4), not the notes
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #shrinks the time again (4 -> 2)
    nn.Flatten(),
    nn.Linear(32 * picture_train.shape[1] * 2, 128), #picture_train.shape[1] is the number of notes (12 or 84)
    nn.ReLU(),
    nn.Linear(128, len(root_names)) #1 score per root
)

root_optimizer = torch.optim.Adam(cnn_root.parameters(), lr=0.0001)

print("training the root model")
for epoch in range(epochs):
    cnn_root.train()
    for batch_X, batch_y in root_loader:
        root_optimizer.zero_grad() #clears the gradients from the last batch
        output = cnn_root(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        root_optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 1 == 0:
        cnn_root.eval()
        with torch.no_grad():
            val_output = cnn_root(picture_val_tensor)
        val_pred = torch.argmax(val_output, dim=1)
        val_accuracy = accuracy_score(y_val_root_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#-------the quality model-------
quality_data = TensorDataset(picture_train_tensor, y_train_quality_tensor)
quality_loader = DataLoader(quality_data, batch_size=64, shuffle=True)

cnn_quality = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #only shrinks the time (8 -> 4), not the notes
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d((1, 2)), #shrinks the time again (4 -> 2)
    nn.Flatten(),
    nn.Linear(32 * picture_train.shape[1] * 2, 128), #picture_train.shape[1] is the number of notes (12 or 84)
    nn.ReLU(),
    nn.Linear(128, len(quality_names)) #1 score per quality
)

quality_optimizer = torch.optim.Adam(cnn_quality.parameters(), lr=0.001)

print("\ntraining the quality model")
for epoch in range(epochs):
    cnn_quality.train()
    for batch_X, batch_y in quality_loader:
        quality_optimizer.zero_grad() #clears the gradients from the last batch
        output = cnn_quality(batch_X) #runs the batch through the model
        loss = loss_function(output, batch_y) #how wrong was it
        loss.backward() #works out the gradients
        quality_optimizer.step() #updates the weights

    #every 10 epochs check how its doing on the validation set
    if (epoch + 1) % 10 == 0:
        cnn_quality.eval()
        with torch.no_grad():
            val_output = cnn_quality(picture_val_tensor)
        val_pred = torch.argmax(val_output, dim=1)
        val_accuracy = accuracy_score(y_val_quality_numbers, val_pred.numpy())
        print(f"epoch {epoch + 1}: loss {loss.item():.3f}, val accuracy {val_accuracy:.3f}")


#-------now test both of them on the test set-------
cnn_root.eval()
cnn_quality.eval()
with torch.no_grad():
    root_output = cnn_root(picture_test_tensor)
    quality_output = cnn_quality(picture_test_tensor)
root_pred_numbers = torch.argmax(root_output, dim=1)
quality_pred_numbers = torch.argmax(quality_output, dim=1)

#turn the numbers back into the names, and stick them back together into full chords
root_pred = []
quality_pred = []
combined_pred = []
for i in range(len(y_test)):
    root = root_names[root_pred_numbers[i]]
    quality = quality_names[quality_pred_numbers[i]]
    root_pred.append(root)
    quality_pred.append(quality)
    combined_pred.append(root + ":" + quality)

print("\nCNN (2 models)")
print(f"root accuracy: {accuracy_score(y_test_root, root_pred):.3f}")
print(f"quality accuracy: {accuracy_score(y_test_quality, quality_pred):.3f}")
print(f"combined: {accuracy_score(y_test, combined_pred):.3f}") #only counts as correct if both the root and the quality are right

training the root model
epoch 1: loss 2.135, val accuracy 0.441
epoch 2: loss 1.603, val accuracy 0.588
epoch 3: loss 1.248, val accuracy 0.665
epoch 4: loss 0.907, val accuracy 0.704
epoch 5: loss 0.893, val accuracy 0.719
epoch 6: loss 0.560, val accuracy 0.750
epoch 7: loss 0.627, val accuracy 0.781
epoch 8: loss 0.386, val accuracy 0.808
epoch 9: loss 0.581, val accuracy 0.816
epoch 10: loss 0.622, val accuracy 0.848
epoch 11: loss 0.330, val accuracy 0.853
epoch 12: loss 0.302, val accuracy 0.856
epoch 13: loss 0.302, val accuracy 0.874
epoch 14: loss 0.413, val accuracy 0.870
epoch 15: loss 0.339, val accuracy 0.885
epoch 16: loss 0.158, val accuracy 0.893
epoch 17: loss 0.408, val accuracy 0.899
epoch 18: loss 0.356, val accuracy 0.896
epoch 19: loss 0.272, val accuracy 0.909
epoch 20: loss 0.225, val accuracy 0.911
epoch 21: loss 0.206, val accuracy 0.910
epoch 22: loss 0.188, val accuracy 0.910
epoch 23: loss 0.256, val accuracy 0.913
epoch 24: loss 0.240, val accuracy 0.909
e

KeyboardInterrupt: 